# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Formulation de vos hypothèses *(À compléter)* :**
- **Test #1 — Mann-Whitney U (pluie vs sec, courses Lyft) :** $H_0 =$ `la distribution du prix au mile est la même par temps sec et sous la pluie` / $H_1 =$ `elle diffère` *(Justification : 2 groupes indépendants, variable très asymétrique (skewness > 0) → test non paramétrique sur les rangs ; on se limite à Lyft car Uber n'applique jamais de majoration dans ces données)*
- **Test #2 — χ² d'indépendance (majoration × pluie, Lyft) :** $H_0 =$ `la majoration (is_surge) est indépendante de la pluie (is_pluie)` / $H_1 =$ `elles sont liées` *(Justification : 2 variables qualitatives binaires → tableau de contingence 2×2 ; taille d'effet par le V de Cramér)*
- **Test #3 — Kruskal-Wallis (5 tranches horaires) :** $H_0 =$ `le prix au mile a la même distribution dans les 5 tranches horaires` / $H_1 =$ `au moins une tranche diffère` *(Justification : k = 5 groupes, variable asymétrique → généralisation non paramétrique de Mann-Whitney ; taille d'effet ε²)*
- **Test #4 — Spearman (distance ↔ prix) :** $H_0 : \rho = 0$ / $H_1 : \rho \neq 0$ *(Justification : lien monotone entre 2 variables quantitatives non gaussiennes, robuste aux prix extrêmes)*

> **Attention aux très grands échantillons :** avec des dizaines de milliers de courses, la moindre différence devient « significative » (p ≈ 0). On conclut donc aussi avec une **taille d'effet**, qui dit si la différence est grande ou négligeable en pratique.

In [2]:
SILVER_MAIN_PATH = "/lakehouse/silver/rides"

# Projection des seules colonnes utiles + échantillon aléatoire de 10 % côté Spark, puis rapatriement vers Pandas
silver = spark.read.format("delta").load(SILVER_MAIN_PATH)
df_sample = (
    silver.select("cab_type", "price", "distance", "prix_par_mile", "is_surge", "is_pluie", "tranche_horaire")
    .sample(fraction=0.10, seed=42)
    .toPandas()
)
print(f"Échantillon : {len(df_sample):,} courses ({df_sample['cab_type'].value_counts().to_dict()})")
assert df_sample is not None, "TODO : Extrayez les colonnes nécessaires dans `df_sample`."

resultats = []
lyft = df_sample[df_sample["cab_type"] == "Lyft"]

# Test 1 : Mann-Whitney U — prix au mile, pluie vs sec (Lyft)
sec, pluie = lyft.loc[lyft["is_pluie"] == 0, "prix_par_mile"], lyft.loc[lyft["is_pluie"] == 1, "prix_par_mile"]
u, p1 = stats.mannwhitneyu(pluie, sec, alternative="two-sided")
r_rb = 1 - 2 * u / (len(pluie) * len(sec))          # corrélation rang-bisériale (taille d'effet)
resultats.append(("Mann-Whitney U", "prix/mile Lyft : pluie vs sec", u, p1, "rang-bisériale r", r_rb,
                  f"médianes : pluie {pluie.median():.2f} $ / sec {sec.median():.2f} $"))

# Test 2 : Khi-deux — majoration x pluie (Lyft)
table = pd.crosstab(lyft["is_pluie"], lyft["is_surge"])
chi2, p2, ddl, _ = stats.chi2_contingency(table)
v_cramer = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
taux = lyft.groupby("is_pluie")["is_surge"].mean() * 100
resultats.append(("Khi-deux", "majoration x pluie (Lyft)", chi2, p2, "V de Cramér", v_cramer,
                  f"% majorées : pluie {taux.get(1, float('nan')):.2f} % / sec {taux.get(0, float('nan')):.2f} %"))

# Test 3 : Kruskal-Wallis — prix au mile selon les 5 tranches horaires
groupes = [g["prix_par_mile"].values for _, g in df_sample.groupby("tranche_horaire")]
h, p3 = stats.kruskal(*groupes)
k, n = len(groupes), len(df_sample)
eps2 = (h - k + 1) / (n - k)                         # epsilon² (taille d'effet)
med_tranches = df_sample.groupby("tranche_horaire")["prix_par_mile"].median().round(2).to_dict()
resultats.append(("Kruskal-Wallis", "prix/mile selon 5 tranches horaires", h, p3, "epsilon²", eps2,
                  f"médianes : {med_tranches}"))

# Test 4 : Spearman — distance vs prix
rho, p4 = stats.spearmanr(df_sample["distance"], df_sample["price"])
r_p, _ = stats.pearsonr(df_sample["distance"], df_sample["price"])
resultats.append(("Spearman", "distance <-> prix", rho, p4, "rho de Spearman", rho, f"Pearson r = {r_p:.3f}"))

tests_pd = pd.DataFrame(resultats, columns=["test", "hypothese", "statistique", "p_value", "mesure_effet", "taille_effet", "detail"])
tests_pd["decision_5pct"] = np.where(tests_pd["p_value"] < 0.05, "H0 rejetée", "H0 conservée")
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 90)
print(tests_pd.round(4).to_string(index=False))

# Sauvegarde des résultats en Gold pour le dashboard
GOLD_TESTS_PATH = "/lakehouse/gold/tests_statistiques"
(spark.createDataFrame(tests_pd.astype({"statistique": float, "p_value": float, "taille_effet": float}))
 .coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_TESTS_PATH))

Échantillon : 63,783 courses ({'Uber': 33110, 'Lyft': 30673})


          test                           hypothese  statistique  p_value     mesure_effet  taille_effet                                                                                                                                                                    detail decision_5pct
Mann-Whitney U       prix/mile Lyft : pluie vs sec 6.673356e+07   0.3317 rang-bisériale r       -0.0085                                                                                                                                      médianes : pluie 7.92 $ / sec 7.92 $  H0 conservée
      Khi-deux           majoration x pluie (Lyft) 9.224000e-01   0.3368      V de Cramér        0.0055                                                                                                                                    % majorées : pluie 6.55 % / sec 6.93 %  H0 conservée
Kruskal-Wallis prix/mile selon 5 tranches horaires 2.521300e+00   0.6408         epsilon²       -0.0000 médianes : {'1. Nuit (0h-6h)': 7

### Conclusions des tests (échantillon aléatoire de 63 783 courses, seuil 5 %)

| Test | p-value | Taille d'effet | Conclusion |
|---|---|---|---|
| **Mann-Whitney U** — prix au mile Lyft, pluie vs sec | 0,33 | r = −0,008 (nul) | **H0 conservée** : même prix médian au mile sous la pluie et par temps sec (7,92 $ / 7,92 $) |
| **χ²** — majoration × pluie (Lyft) | 0,34 | V de Cramér = 0,006 (nul) | **H0 conservée** : la majoration est **indépendante de la pluie** (6,55 % vs 6,93 %) |
| **Kruskal-Wallis** — prix au mile selon 5 tranches horaires | 0,64 | ε² ≈ 0 | **H0 conservée** : les médianes vont de 7,38 $ (soirée) à 7,58 $ (pointe du matin), un écart négligeable |
| **Spearman** — distance ↔ prix | < 0,0001 | ρ = 0,33 (modéré) | **H0 rejetée** : lien monotone **positif modéré** (Pearson r = 0,34) |

**Lecture métier :**
- Même avec plus de 60 000 observations, où la moindre différence serait « significative », **la pluie et l'heure n'ont aucun effet mesurable** sur le prix. C'est cohérent avec la nature du dataset : des **prix simulés** obtenus en interrogeant les API à intervalles réguliers, et non de vraies réservations. Dans ces conditions, la demande réelle (qui crée la majoration) n'est pas observée.
- Le prix dépend d'abord de la **gamme** (boxplot du Jour 2) puis de la **distance** (ρ = 0,33). Ce lien n'est que modéré, car à distance égale un Shared et un Black SUV n'ont pas du tout le même prix.

## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [3]:
GOLD_SEGMENTS_PATH = "/lakehouse/gold/segments_trajets"  # entité segmentée : le trajet (quartier de départ -> d'arrivée)

from sklearn.metrics import silhouette_score

# 1. Agrégation PySpark : 1 ligne par trajet, 5 features comportementales
entities_pd = (
    silver.groupBy("trajet", "source", "destination")
    .agg(
        F.count("*").alias("nb_courses"),
        F.percentile_approx("distance", 0.5).alias("distance_mediane"),
        F.percentile_approx("price", 0.5).alias("prix_median"),
        F.percentile_approx("prix_par_mile", 0.5).alias("prix_mile_median"),
        F.round(100 * F.avg("is_surge"), 3).alias("pct_majorees"),
    )
    .toPandas()
)
print(f"{len(entities_pd)} trajets à segmenter")
assert entities_pd is not None, "TODO : Construisez le DataFrame d'entités agrégées `entities_pd`."

# 2. Prétraitement : log1p sur les features positives asymétriques, puis centrage-réduction
FEATURES = ["nb_courses", "distance_mediane", "prix_median", "prix_mile_median", "pct_majorees"]
X = StandardScaler().fit_transform(np.log1p(entities_pd[FEATURES]))

# Choix de k : score de silhouette (plus haut = groupes mieux séparés)
scores = {k: silhouette_score(X, KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(X)) for k in range(2, 7)}
print("Silhouette par k :", {k: round(s, 3) for k, s in scores.items()})
K = max(scores, key=scores.get)
print(f"k retenu = {K}")
entities_pd["cluster_id"] = KMeans(n_clusters=K, n_init=20, random_state=42).fit_predict(X)

# 3. Interprétation : médianes par cluster, puis nom métier lisible
profil = entities_pd.groupby("cluster_id")[FEATURES].median()
profil["nb_trajets"] = entities_pd.groupby("cluster_id").size()
print(profil.round(2).to_string())

def niveau(serie, valeur, mots):
    rang = serie.rank(method="dense").loc[valeur]
    return mots[0] if rang == 1 else (mots[-1] if rang == serie.nunique() else mots[1])

noms = {}
for cid, row in profil.iterrows():
    dist = niveau(profil["distance_mediane"], cid, ["Courts", "Moyens", "Longs"])
    prix = niveau(profil["prix_mile_median"], cid, ["prix au mile bas", "prix au mile moyen", "prix au mile élevé"])
    noms[cid] = f"{dist} trajets · {prix}"
# si deux clusters ont le même nom, on les distingue par le taux de majoration
for nom in set(noms.values()):
    ids = [c for c, n in noms.items() if n == nom]
    if len(ids) > 1:
        for c in ids:
            noms[c] += f" · {profil.loc[c, 'pct_majorees']:.1f} % majorées"
entities_pd["segment"] = entities_pd["cluster_id"].map(noms)
print(entities_pd.groupby("segment")[FEATURES].median().round(2).to_string())

assert "segment" in entities_pd.columns, "TODO : Ajoutez une colonne lisible `segment` décrivant chaque cluster."

# TODO 3 : Sauvegarder en Delta Lake sur HDFS dans `GOLD_SEGMENTS_PATH`
(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)
print(f"Table de segmentation enregistrée dans {GOLD_SEGMENTS_PATH} ({len(entities_pd):,} entités) !")

72 trajets à segmenter


Silhouette par k : {2: 0.387, 3: 0.303, 4: 0.313, 5: 0.321, 6: 0.318}
k retenu = 2
            nb_courses  distance_mediane  prix_median  prix_mile_median  pct_majorees  nb_trajets
cluster_id                                                                                       
0               8838.0              2.67         16.5              6.11          4.27          49
1               8802.0              1.09         11.0             10.38          1.73          23
                                     nb_courses  distance_mediane  prix_median  prix_mile_median  pct_majorees
segment                                                                                                       
Courts trajets · prix au mile élevé      8802.0              1.09         11.0             10.38          1.73
Longs trajets · prix au mile bas         8838.0              2.67         16.5              6.11          4.27


Table de segmentation enregistrée dans /lakehouse/gold/segments_trajets (72 entités) !


### Interprétation de la segmentation (`/lakehouse/gold/segments_trajets`)

- **Choix de k :** le score de silhouette est le plus élevé pour **k = 2** (0,387, contre 0,30 à 0,32 pour k = 3 à 6) : les trajets se séparent naturellement en **2 familles**.

| Segment | Trajets | Distance médiane | Prix médian | Prix médian au mile | % majorées |
|---|---|---|---|---|---|
| **Courts trajets · prix au mile élevé** | 23 | 1,09 mile | 11,00 $ | **10,38 $** | 1,73 % |
| **Longs trajets · prix au mile bas** | 49 | 2,67 miles | 16,50 $ | **6,11 $** | 4,27 % |

- **Lecture métier :** les courses courtes paient surtout la **prise en charge** (forfait minimum), d'où un prix au mile 1,7 fois plus élevé. Les longs trajets coûtent plus cher au total mais moins cher au mile, et sont **plus souvent majorés** (4,3 % contre 1,7 %).
- Le volume de courses (≈ 8 800 par trajet dans les deux segments) ne discrimine pas les trajets, car la collecte est régulière. La segmentation repose bien sur la **distance** et le **prix au mile**, comme l'axe PC1 de l'ACP du Jour 2.

## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

### Notre application `app/dashboard.py` (http://localhost:8501)

- **Architecture respectée :** l'application ne lit **que** les 8 tables de `/lakehouse/gold/` (`resume_distributions`, `pareto_trajets`, `evolution_journaliere`, `profil_horaire`, `effet_meteo`, `offre_moins_chere_par_quartier`, `segments_trajets`, `tests_statistiques`), jamais les CSV ni la Silver.
- **Indicateurs clés :** courses analysées, **prix médian**, prix moyen (avec son écart à la médiane), **IQR**, skewness.
- **Filtres :** compagnie (`st.multiselect`), nombre de trajets du Pareto (`st.slider`), gamme du duel Uber / Lyft (`st.selectbox`).
- **5 onglets, 6 graphiques Plotly :** Pareto des trajets (barres + % cumulé), prix médian par heure de Boston, évolution journalière (avec alerte sur les trous de collecte), effet de la pluie par gamme, duel du moins cher au mile par quartier, nuage des segments K-Means. S'y ajoutent le tableau des segments et celui des tests.
- **Rigueur explicative :** sous chaque analyse, un `st.expander("Requête SQL & Justification statistique")` affiche la requête Spark SQL et le choix des indicateurs (médiane / IQR, Pareto, `LAG`, `ROW_NUMBER`, tests non paramétriques, log1p + StandardScaler).

In [4]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")

=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===


 - /lakehouse/gold/effet_meteo               :        22 lignes


 - /lakehouse/gold/evolution_journaliere     :        18 lignes


 - /lakehouse/gold/offre_moins_chere_par_quartier :        72 lignes


 - /lakehouse/gold/pareto_trajets            :        72 lignes


 - /lakehouse/gold/profil_horaire            :        48 lignes


 - /lakehouse/gold/resume_distributions      :         3 lignes


 - /lakehouse/gold/segments_trajets          :        72 lignes


 - /lakehouse/gold/tests_statistiques        :         4 lignes
